## Start session

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!chmod +x install.sh
! ./install.sh > /dev/null 2>&1

In [ ]:
import os
import boto3
import subprocess

from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import torch

from sklearn.model_selection import train_test_split

import ignite_classes as ic

## Download data

In [ ]:
# Bucket ids
mybucket = "maximelenormand"
key='DQV2PBOU23BJBMKO243K'
secret='xZO1qnt7+aSwY+IKjQw+g7Hj6xKugc7EQFSj9PEL'
token='eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhY2Nlc3NLZXkiOiJEUVYyUEJPVTIzQkpCTUtPMjQzSyIsImFsbG93ZWQtb3JpZ2lucyI6WyIqIl0sImF1ZCI6WyJtaW5pby1kYXRhbm9kZSIsIm9ueXhpYSIsImFjY291bnQiXSwiYXV0aF90aW1lIjoxNzc0NTE0MjkwLCJhenAiOiJvbnl4aWEiLCJjbmYiOnsiamt0IjoiYVNtNXRyRjZQamZxbHBrcHV5X1hrZTY1UnRDU3VSaWdEbzRKZF9BR2tyQSJ9LCJlbWFpbCI6Im1heGltZS5sZW5vcm1hbmRAaW5yYWUuZnIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZXhwIjoxNzc1MTE5Mzc0LCJmYW1pbHlfbmFtZSI6Ikxlbm9ybWFuZCIsImdpdmVuX25hbWUiOiJNYXhpbWUiLCJncm91cHMiOlsiVVNFUl9PTllYSUEiXSwiaWF0IjoxNzc0NTE0NTczLCJpc3MiOiJodHRwczovL2F1dGgubGFiLnNzcGNsb3VkLmZyL2F1dGgvcmVhbG1zL3NzcGNsb3VkIiwianRpIjoib25ydHJ0OjQzNmZjYzlkLTFkMDUtNGQwYS1mYjJiLWM4NWFmYzUxYzM0YSIsIm5hbWUiOiJNYXhpbWUgTGVub3JtYW5kIiwicG9saWN5Ijoic3Rzb25seSIsInByZWZlcnJlZF91c2VybmFtZSI6Im1heGltZWxlbm9ybWFuZCIsInJlYWxtX2FjY2VzcyI6eyJyb2xlcyI6WyJvZmZsaW5lX2FjY2VzcyIsInVtYV9hdXRob3JpemF0aW9uIiwiZGVmYXVsdC1yb2xlcy1zc3BjbG91ZCJdfSwicmVzb3VyY2VfYWNjZXNzIjp7ImFjY291bnQiOnsicm9sZXMiOlsibWFuYWdlLWFjY291bnQiLCJtYW5hZ2UtYWNjb3VudC1saW5rcyIsInZpZXctcHJvZmlsZSJdfX0sInJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNzcGNsb3VkIl0sInNjb3BlIjoib3BlbmlkIHByb2ZpbGUgZ3JvdXBzIGVtYWlsIiwic2lkIjoiYWZlOTg1ZjMtMWQzOS0xNDEyLTZiMzUtYmM1NDkwMTJmMzM3Iiwic3ViIjoiM2M5MDk3MWMtNDEwMC00NmNhLWI3OTYtMTg5MGU0N2NhYWZkIiwidHlwIjoiRFBvUCJ9.sQWnf0mvrseOTJwfQmJQgWHe5XI6G2HggeRF0xPIk4MghMv9vF54YY0-AaqBXAVM_ez9_pn3btVAdJcO0GCT7Q'

s3 = boto3.client("s3",endpoint_url = 'https://minio.lab.sspcloud.fr',
                  aws_access_key_id = key, 
                  aws_secret_access_key = secret, 
                  aws_session_token = token)

# Check connection
listobjbucket = s3.list_objects_v2(Bucket=mybucket, MaxKeys=5)
if 'Contents' in listobjbucket:
    print("Connection OK!\nFiles:")
    for obj in listobjbucket['Contents']:
        print("-", obj['Key'])

# Download data.zip if needed
if not os.path.exists("/home/onyxia/work/data.zip"):
    s3.download_file(mybucket, "data.zip", "data.zip")

# Unzip if needed
if not os.path.exists("/home/onyxia/work/data"):
    subprocess.run(["unzip", "data.zip"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)
    print("Done!")


## Load data

In [ ]:
binary_columns_tot = [
    "human",
    "anthropic",
    "vegetation",
    "rock",
    "snow",
    "water",
    "zoom",
    "animal",
    "no",
]
binary_columns_todrop = ["animal", "no", "water", "zoom"]
binary_columns = ["human", "anthropic", "vegetation", "rock", "snow"]
csv_raw = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv")).drop(
    binary_columns_todrop, axis=1
)
csv_raw

In [ ]:
csv_raw.sum()

In [ ]:
n = 2000

rng = np.random.default_rng(42)

# ------------------------------------------------------------------ #
# Compute weights from imbalance in the original dataframe
# ------------------------------------------------------------------ #
# p = proportion of 1s; distance from 0.5 ranges from 0 (perfect balance)
# to 0.5 (all 0s or all 1s). We map it to a weight >= 1.
# weight = 1 + k * (|p - 0.5| / 0.5)  with k controlling the max weight.
k = 4  # max additional weight on top of the baseline 1
weights = {}
# print("\nAuto-computed column weights:")
for col in binary_columns:
    p = csv_raw[col].mean()
    imbalance = abs(p - 0.5) / 0.5  # 0 = perfectly balanced, 1 = fully skewed
    weights[col] = 1 + k * imbalance
    # print(f"  {col}: proportion of 1s = {p:.3f}, weight = {weights[col]:.2f}")

# ------------------------------------------------------------------ #
# Greedy balanced selection
# ------------------------------------------------------------------ #
df_shuffled = csv_raw.sample(frac=1, random_state=int(rng.integers(1e6))).reset_index(
    drop=True
)

selected_indices = []
counts = {col: {0: 0, 1: 0} for col in binary_columns}
target_per_class = n // 2

for _, row in df_shuffled.iterrows():
    if len(selected_indices) >= n:
        break

    score = 0
    for col in binary_columns:
        w = weights[col]
        val = int(row[col])
        current = counts[col][val]
        current_opposite = counts[col][1 - val]

        if current < target_per_class:
            score += 1 * w
        elif current >= target_per_class and current_opposite < target_per_class:
            score -= 1 * w

    if score >= 0:
        selected_indices.append(row.name)
        for col in binary_columns:
            counts[col][int(row[col])] += 1

csv = df_shuffled.loc[selected_indices].reset_index(drop=True)

pd.DataFrame(
    data={
        "var": [col for col in binary_columns],
        "per": [csv[col].mean() * 100 for col in binary_columns],
        "qtt": [sum(csv[col]) for col in binary_columns],
        "all": len(csv),
    }
).sort_values("per", ascending=False)

## Check labels

In [ ]:
pprint(csv.columns[2:])

## Test Datasets

In [ ]:
path_to_images = Path(".").joinpath("data").joinpath("images")
path_to_images.is_dir()

In [ ]:
dataset = ic.FldDataset(data = csv, train_mode=True,test_mode=True)

In [ ]:
pprint(dataset.transform)
rnd_data = dataset[randint(0, len(dataset) - 1)]
pprint(rnd_data["labels"])
plt.imshow(rnd_data["image"])

In [ ]:
plt.imshow(dataset[100]["image"])

## Train

### Split dataset

In [ ]:
csv_strat = csv.copy()
csv_strat["strat"] = ""
for col in csv.columns[2:]:
    print(col)
    csv_strat["strat"] += csv_strat[col].astype(str)

pprint(csv_strat.strat.value_counts())

csv_strat

In [ ]:
csv_strat_balanced = csv_strat.copy()
csv_strat_count = pd.DataFrame(csv_strat.strat.value_counts()).reset_index()
strat_tokeep = csv_strat_count[csv_strat_count['count'] >= 10]
csv_strat_balanced = csv_strat_balanced[csv_strat_balanced['strat'].isin(strat_tokeep['strat'])]
pprint(csv_strat_balanced.strat.value_counts())

In [ ]:
trainval, test = train_test_split(csv_strat_balanced, test_size = 0.15, random_state = 42, stratify=csv_strat_balanced["strat"])
train, val = train_test_split(trainval, test_size = 0.18, stratify=trainval["strat"])

train = train.drop("strat", axis=1)
val = val.drop("strat", axis=1)
test = test.drop("strat", axis=1)

for n,d in [("train",train), ("val",val), ("test",test)]:
    print(n, d.shape)

### Train 10 models for each selected backbone

In [ ]:
for backbone in [
    "hf_swt_t",
    # "hf_resnet",
    # "hf_cnx_t",
    # "hf_vit_g16",
]:
    for _ in list(range(1)):
        ic.train_model(
            train_data=train,
            val_data=val,
            batch_size=32,
            max_epochs=20,
            image_size=224,
            run_owner="moi",
            exp_name="image_labeller",
            backbone=backbone,
            loss_name="bce",
            loss_params={"alpha": 0.5, "gamma": 1},
            device=ic.get_device(),
            checkpoints_n_saved=1,
            learning_rate=0.00001,
            early_stoper_patience=10,
            early_stoper_min_delta=0.001,
            use_lr_finder=False,
            lr_scheduler_step=10,
            lr_scheduler_gamma=0.85,
            print_steps="print",
            log_progress=True,
            plot_loss=False,
            num_workers=10,
        )

### Load experiment data

WARNING: All fields are stored as strings and need to be converted to other formats if needed

In [ ]:
runs = (
    mlflow.search_runs(
        search_all_experiments=True,
        experiment_names=["image_labeller"],
        order_by=[f"params.F1_weighted_avg DESC"],
    )
    .assign( # Cast metrics to float numbers
        **{
            k: lambda x: x[k].astype(np.float32)
            for k in [
                "params.F1_weighted_avg",
                "params.F1_samples_avg",
                "params.F1_macro_avg",
                "params.F1_samples_avg",
                "params.F1_snow",
                "params.F1_vegetation",
                "params.F1_anthropic",
                "params.F1_human",
                "params.F1_rock",
            ]
        }
    )
)
runs

### Display mean and standard deviation for each backbone

In [ ]:
runs.groupby(["params.backbone"]).agg(
    {
        k: ["mean", "std"]
        for k in [
            "params.F1_weighted_avg",
            "params.F1_samples_avg",
            "params.F1_macro_avg",
            "params.F1_samples_avg",
            "params.F1_snow",
            "params.F1_vegetation",
            "params.F1_anthropic",
            "params.F1_human",
            "params.F1_rock",
        ]
    }
).reset_index()

### Select best backbone

In [ ]:
backbone_averages = (
    runs.groupby(["params.backbone"])
    .agg({"params.F1_weighted_avg": "mean"})
    .reset_index()
).sort_values("params.F1_weighted_avg", ascending=False)
backbone_averages

### Load best model

In [ ]:
best_run = (
    runs[runs["params.backbone"] == backbone_averages.iloc[0]["params.backbone"]]
    .sort_values("params.F1_weighted_avg", ascending=False)
    .iloc[0]
)

model = mlflow.pytorch.load_model(
    f"runs:/{best_run.run_id}/model",
    map_location=torch.device(ic.get_device()),
)
model.hr_desc()

### Display validation data for best data

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=val, train_mode=False))["classification_report"]

## End session

### Git

In [ ]:
subprocess.run(["cp", "ignite_classes.py", "AI/ignite_classes.py"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)
subprocess.run(["cp", "ignite_demo.ipynb", "AI/ignite_demo.ipynb"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)
subprocess.run(["cp", "install.", "AI/ignite_demo.ipynb"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

### Bucket

In [ ]:
subprocess.run(["rm", "mlruns.zip"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

subprocess.run(["zip", "-r", "mlruns.zip", "mlruns"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

In [ ]:
s3.upload_file("mlruns.zip", mybucket, "mlruns.zip")
s3.upload_file("mlflow.db", mybucket, "mlflow.db")